# Results analysis — P2P-Thief (anrbj666)

Sensitivity analysis (OAT) over the parameters that shape the pursuit game.
Raw data: `results/experiments/sensitivity.json`; regenerate everything with
`uv run python scripts/run_sensitivity.py`.

## 1. Scent decay — why $\rho = 0.10$ is the game's memory

The trail obeys $\tau_t = \tau_0 (1-\rho)^t$ with $\tau_0 = 0.9$. The
lie-detection floor (0.4) is crossed after
$t^* = \frac{\ln(0.4/0.9)}{\ln(1-\rho)}$ turns — at the fixed $\rho=0.10$
that is $t^* \approx 7.7$: a trail stays *legally readable* for ~7 turns,
long enough to catch lies one full round later, short enough that history
does not drown the present (the book's local-optima argument, ch. 4).

![decay](../assets/sens_decay.png)

In [ ]:
import json
import math
from pathlib import Path

sens = json.loads(Path("../results/experiments/sensitivity.json").read_text())
for rho in (0.05, 0.10, 0.20, 0.30):
    t_star = math.log(0.4 / 0.9) / math.log(1 - rho)
    print(f"rho={rho}: readable for ~{t_star:.1f} turns")

## 2. Hint honesty — deception does not save a weak runner

Blind pursuit (belief-only) vs a random thief across honesty levels
$P(\text{truth}) \in \{0, .25, .5, .75, 1\}$, 15 seeds each. Capture rate
stays in the 0.93–1.0 band **independent of honesty**: the scent evidence
dominates the posterior, so lying only helps a thief whose *movement* also
exploits the belief. This validated our decision to invest in movement
strategy over prompt sophistication (moves are pure Python anyway — rule 25).

![honesty](../assets/sens_honesty.png)

In [ ]:
print(json.dumps(sens["honesty_capture_rate"], indent=2))

## 3. Board size — every added cell is thief territory

Mean turns-to-capture under full information grows superlinearly-ish with
the board side (7.7 → 9.3 → 12.1 for 7/9/11): the state space of the
Dec-POMDP grows as $O(n^4)$ (two positions) times barrier configurations,
while the survival threshold stays 35 — so negotiating a larger board is a
pro-thief move, and our cop should resist raising the minimum.

![board](../assets/sens_board.png)

In [ ]:
print(sens["board_capture_turns"])

## Conclusions

1. Keep $ho=0.10$ (fixed anyway) — the ~7-turn readable window is what
   makes the (1−ρ)·0.9 lie test decisive.
2. Strategy budget goes to movement, not rhetoric: honesty sweeps show the
   verbal layer cannot rescue weak evasion against a belief-driven pursuer.
3. Board-size negotiation is strategic: cop wants 7×7, thief wants bigger.

References: Bernstein et al. (Dec-POMDP complexity); Theraulaz & Bonabeau
(stigmergy); the course rulebook ch. 4–6.